# 第80章 地图图表（Map / Geo）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 17 / 18 步：组合控件、子图并完成交付**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 时间线与甘特图（px.timeline）  →  **本章任务：** 地图图表（Map / Geo）  →  **下一步：** 子图、控件与导出
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

当你想要对比不同城市或国家的销售额时，条形图能列出数字，却很难让人一眼看清“哪些地区更强、哪些更弱”。


## 本章目标

学完本章，你将能够：

- **理解**：理解「地图图表（Map / Geo）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「地图图表（Map / Geo）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「地图图表（Map / Geo）」并读出其中的结论。


## 适用场景

**背景引入**：当你想要对比不同城市或国家的销售额时，条形图能列出数字，却很难让人一眼看清“哪些地区更强、哪些更弱”。地图图表把每个位置连同对应指标画在同一张图上，视觉上一扫就能判断空间分布和大小差异，是分析市场布局、区域业绩这类问题的直接选择。（好比一张给数据“贴标签”的世界地图：图上每个地区既是一个位置，又被涂上代表大小或强弱的颜色；想画它，先得有能对上的地区名或经纬度（编码），否则地图认不出你说的“广东”是哪个。）

地理位置本身对解释有意义，例如市场、网点或区域指标。


## 数据结构

标准地理编码、名称以及数值指标。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 projection="natural earth" 改为 "orthographic" 或 "mercator"，对比不同地图投影的效果
2. 修改 color_continuous_scale 从 "Blues" 为 "YlOrRd"，观察色盘对指标差异的表达
3. 添加 hover_data 显示增长率等补充字段，说明悬浮信息对空间数据解读的作用


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `px.scatter_geo()`、`fig.update_layout()`、`fig.show()` | 地理位置本身对解释有意义，例如市场、网点或区域指标。 | 地理编码无法匹配 |
| 进阶变体 | `px.choropleth()`、`fig.update_layout()`、`fig.show()` | 在基础图表上增加分组、注释、布局或交互 | 大区域视觉面积造成偏见 |
| 关键参数 | `locations` | 地理编码 | 地理编码无法匹配 |
| 关键参数 | `locationmode` | 编码类型 | 大区域视觉面积造成偏见 |
| 关键参数 | `projection` | 投影 | 无空间意义的数据强行使用地图 |
| 关键参数 | `scope` | 区域范围 | 地理编码无法匹配 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-80 -->
### 数学推导｜地图比较应优先使用率而不是总量

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜区分事件数与机会数。** 地区 $i$ 有事件数 $count_i$，但产生事件的暴露量是 $exposure_i$。

**第 2 步｜除以暴露量消除规模影响。** 原始率为 $count_i/exposure_i$。

**第 3 步｜乘统一倍率提高可读性。** 每万人发生数为

$$
rate_i=\frac{count_i}{exposure_i}\times10{,}000
$$

倍率只改变显示单位，不改变地区排序；小分母却会放大随机波动。

**把上面的关系收束为本章计算式：**

$$
rate_i=\frac{count_i}{exposure_i}\times k
$$

**符号解释：** $exposure_i$ 是人口、客户数或业务机会数，$k$ 是统一倍率。

**代码对应：** 将事件数与暴露量合并后计算标准化率，再映射颜色或大小。

**使用边界：** 行政区面积会制造视觉偏差；缺失地区和小分母率值需要标注。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(
    f'Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行'
)


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.scatter_geo(
    countries,
    locations="country",
    locationmode="country names",
    size="sales",
    color="growth",
    hover_name="market",
    projection="natural earth",
    color_continuous_scale="Blues",
    title="Gapminder：人均GDP与预期寿命",
)
fig.update_layout(coloraxis_colorbar_title="预期寿命")
fig.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


### 练一练：换一种投影，观察地图的变化

上面这张基础图表用 `projection="natural earth"` 把整个世界画成平面地图。请把投影改成 `"orthographic"`（像从太空看地球那样，球面视角）或 `"mercator"`（常见的等距柱状投影），重新运行并比较：同一组销售点在地图上的排布发生了什么变化？

提示：练习单元格里已经搭好脚手架，只需要把 `projection` 的值填进去即可；对照右侧答案能确认你的写法。


In [ ]:
try:
    pass
    # 请在下方填写代码：把 projection 从 "natural earth" 改为 "orthographic" 或 "mercator"

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = px.choropleth(
    countries,
    locations="country",
    locationmode="country names",
    color="sales",
    hover_name="market",
    hover_data={"growth": ":.1f"},
    projection="natural earth",
    color_continuous_scale="YlGnBu",
    title="Gapminder：各国人均GDP",
)
fig.update_layout(coloraxis_colorbar_title="人均GDP")
fig.show()


## 参数说明

- locations：地理编码
- locationmode：编码类型
- projection：投影
- scope：区域范围


## 结果解读

结合位置和指标读取空间模式；地图面积不能替代数值比较。


## 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
_demo_fig = px.bar(report, x="region", y="sales", title="地区销售额")
_demo_fig.show()


### 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
_demo_fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
_demo_fig.update_traces(textposition="outside")
_demo_fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
_demo_fig.show()


### 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 地理编码无法匹配
- 大区域视觉面积造成偏见
- 无空间意义的数据强行使用地图


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：换一种投影，观察地图形态的变化
    # 【目标】不同投影会改变地图的形变，练习体会投影对读图的影响。
    import plotly.express as px

    # 起点示例(已可运行)：projection 换成 mercator。
    fig = px.scatter_geo(
        countries,
        locations="country",
        locationmode="country names",
        size="sales",
        color="growth",
        hover_name="market",
        projection="mercator",
        color_continuous_scale="Blues",
        title="Gapminder：人均GDP与预期寿命（墨卡托）",
    )
    fig.update_layout(coloraxis_colorbar_title="预期寿命")
    fig.show()

    # ---- 反思记录：不同投影下，高纬度地区面积是否失真 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用Geo地图表达国家或地区的空间位置、规模和差异。


### 你已经掌握

- 判断地图图表（Map / Geo）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `locations` | 地理编码 |
| `locationmode` | 编码类型 |
| `projection` | 投影 |
| `scope` | 区域范围 |


### 需要注意

- 地理编码无法匹配
- 大区域视觉面积造成偏见
- 无空间意义的数据强行使用地图


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案：把投影改为 orthographic
fig = px.scatter_geo(
    countries,
    locations="country",
    locationmode="country names",
    size="sales",
    color="growth",
    hover_name="market",
    projection="orthographic",
    color_continuous_scale="Blues",
    title="Gapminder：人均GDP与预期寿命（orthographic）",
)
fig.update_layout(coloraxis_colorbar_title="预期寿命")


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
map_points = pd.DataFrame(
    {
        "city": ["上海", "广州", "北京", "成都"],
        "lat": [31.23, 23.13, 39.90, 30.57],
        "lon": [121.47, 113.26, 116.40, 104.07],
        "sales": [320, 250, 280, 190],
    }
)
fig = px.scatter_geo(
    map_points,
    lat="lat",
    lon="lon",
    size="sales",
    hover_name="city",
    projection="natural earth",
    title="国内城市销售点位",
)
fig.update_geos(lataxis_range=[15, 55], lonaxis_range=[70, 140])
fig.show()
